# Smoke test (yolo11n, 3 epochs, imgsz=320)

End-to-end sanity check that takes ~5 min on a T4. Use this before kicking off a real
training run. Verifies: Drive mount, repo clone, dataset unzip + sha256, training
loop, run_meta.json write, atomic copy to Drive, and test-split eval.

**One-time setup:** in Colab, click the 🔑 key icon (left sidebar) → add a secret
named `GITHUB_TOKEN` with a fine-grained PAT that has read access to this repo,
and toggle 'Notebook access' on.

In [ ]:
REPO_URL    = "https://github.com/tahmid013/yolo.git"
REPO_BRANCH = "main"
DATASET_VERSION = "v1"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os, shutil, subprocess
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Set the GITHUB_TOKEN secret in the 🔑 Secrets panel (left sidebar).')

if os.path.isdir('/content/code'):
    shutil.rmtree('/content/code')

# Inject token only inside the subprocess args — never printed.
url = REPO_URL.replace('https://', f'https://{token}@')
subprocess.run(
    ['git', 'clone', '--quiet', '--branch', REPO_BRANCH, url, '/content/code'],
    check=True,
)
os.chdir('/content/code')
print('cloned at:', subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())

!pip install -q -r requirements.txt

In [ ]:
from pipeline.dataset import ensure_dataset
from pipeline import paths
data_yaml = ensure_dataset(
    zip_path=paths.dataset_zip(DATASET_VERSION),
    meta_path=paths.dataset_meta(DATASET_VERSION),
    target=paths.LOCAL_DATASET,
)

In [ ]:
from pipeline.train import run as train_run
run_dir = train_run(
    model='yolo11n',
    config='configs/smoke.yaml',
    drive_runs_dir=paths.RUNS_DIR,
    local_runs_dir=paths.LOCAL_RUNS,
    data_yaml=data_yaml,
    dataset_meta_path=paths.dataset_meta(DATASET_VERSION),
    base_config='configs/base.yaml',
)
print('Smoke run on Drive:', run_dir)

In [ ]:
from pipeline.evaluate import run as eval_run
eval_run(run_dir=run_dir, data_yaml=data_yaml, drive_runs_dir=paths.RUNS_DIR)